# Task 2: End-to-End ML Pipeline with Scikit-learn Pipeline API

## Objective
Build a reusable and production-ready machine learning pipeline for predicting customer churn using the Telco Churn Dataset.

## Skills Demonstrated
- ML pipeline construction with scikit-learn
- Hyperparameter tuning with GridSearchCV
- Model export and reusability using joblib
- Production-ready practices

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_auc_score, roc_curve
)
import joblib
import warnings
warnings.filterwarnings('ignore')

## 1. Dataset Loading & Preprocessing

Load and explore the Telco Customer Churn dataset.

In [ ]:
# Load the Telco Churn Dataset
# Using a public URL for the dataset or creating sample data
url = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"

try:
    df = pd.read_csv(url)
    print("Dataset loaded successfully from URL.")
except:
    # Fallback: Generate synthetic data matching the Telco Churn structure
    print("Could not load from URL. Generating synthetic dataset...")
    np.random.seed(42)
    n_samples = 5000
    
    df = pd.DataFrame({
        'customerID': [f'ID-{i:05d}' for i in range(n_samples)],
        'gender': np.random.choice(['Male', 'Female'], n_samples),
        'SeniorCitizen': np.random.choice([0, 1], n_samples, p=[0.84, 0.16]),
        'Partner': np.random.choice(['Yes', 'No'], n_samples),
        'Dependents': np.random.choice(['Yes', 'No'], n_samples, p=[0.3, 0.7]),
        'tenure': np.random.randint(1, 73, n_samples),
        'PhoneService': np.random.choice(['Yes', 'No'], n_samples, p=[0.9, 0.1]),
        'MultipleLines': np.random.choice(['Yes', 'No', 'No phone service'], n_samples),
        'InternetService': np.random.choice(['DSL', 'Fiber optic', 'No'], n_samples, p=[0.34, 0.44, 0.22]),
        'OnlineSecurity': np.random.choice(['Yes', 'No', 'No internet service'], n_samples),
        'OnlineBackup': np.random.choice(['Yes', 'No', 'No internet service'], n_samples),
        'DeviceProtection': np.random.choice(['Yes', 'No', 'No internet service'], n_samples),
        'TechSupport': np.random.choice(['Yes', 'No', 'No internet service'], n_samples),
        'StreamingTV': np.random.choice(['Yes', 'No', 'No internet service'], n_samples),
        'StreamingMovies': np.random.choice(['Yes', 'No', 'No internet service'], n_samples),
        'Contract': np.random.choice(['Month-to-month', 'One year', 'Two year'], n_samples, p=[0.55, 0.21, 0.24]),
        'PaperlessBilling': np.random.choice(['Yes', 'No'], n_samples, p=[0.6, 0.4]),
        'PaymentMethod': np.random.choice(
            ['Electronic check', 'Mailed check', 'Bank transfer (automatic)', 'Credit card (automatic)'], 
            n_samples
        ),
        'MonthlyCharges': np.random.uniform(18.0, 120.0, n_samples),
        'TotalCharges': np.random.uniform(18.0, 8700.0, n_samples),
        'Churn': np.random.choice(['Yes', 'No'], n_samples, p=[0.27, 0.73])
    })

print(f"Dataset shape: {df.shape}")
print(f"Columns: {len(df.columns)}")

In [ ]:
# Display first few rows
print("\nFirst 5 rows of the dataset:")
df.head()

In [ ]:
# Display dataset info
print("Dataset Info:")
df.info()

In [ ]:
# Statistical summary
print("\nStatistical Summary:")
df.describe()

In [ ]:
# Check for missing values
print("Missing Values:")
missing = df.isnull().sum()
print(missing[missing > 0] if missing.sum() > 0 else "No missing values found.")

# Handle TotalCharges if it's object type (may contain spaces)
if df['TotalCharges'].dtype == 'object':
    df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
    print(f"\nConverted TotalCharges to numeric. Missing after conversion: {df['TotalCharges'].isnull().sum()}")

In [ ]:
# Analyze target variable distribution
print("\nTarget Variable Distribution (Churn):")
churn_counts = df['Churn'].value_counts()
print(churn_counts)
print(f"\nChurn Rate: {(churn_counts['Yes'] / len(df) * 100):.2f}%")

# Visualize
plt.figure(figsize=(6, 4))
sns.countplot(data=df, x='Churn', palette='Set2')
plt.title('Customer Churn Distribution')
plt.show()

## 2. Data Preprocessing & Pipeline Construction

Build preprocessing pipelines for numerical and categorical features.

In [ ]:
# Define features and target
X = df.drop(['Churn', 'customerID'], axis=1)
y = (df['Churn'] == 'Yes').astype(int)  # Convert to binary (1=Churn, 0=No Churn)

# Identify numerical and categorical columns
numerical_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_cols = X.select_dtypes(include=['object']).columns.tolist()

print(f"Numerical features ({len(numerical_cols)}): {numerical_cols}")
print(f"\nCategorical features ({len(categorical_cols)}): {categorical_cols}")

In [ ]:
# Create preprocessing pipelines

# Numerical features: impute missing values with median, then scale
numerical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Categorical features: impute with most frequent, then one-hot encode
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# Combine preprocessing steps
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, numerical_cols),
        ('cat', categorical_transformer, categorical_cols)
    ]
)

print("Preprocessing pipeline created!")
print(f"  - Numerical transformer: Imputer(median) + StandardScaler")
print(f"  - Categorical transformer: Imputer(most_frequent) + OneHotEncoder")

## 3. Train-Test Split

In [ ]:
# Split data into training and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set size: {len(X_train)}")
print(f"Test set size: {len(X_test)}")
print(f"\nTraining churn rate: {(y_train.sum() / len(y_train) * 100):.2f}%")
print(f"Test churn rate: {(y_test.sum() / len(y_test) * 100):.2f}%")

## 4. Model Development & Hyperparameter Tuning

Build complete ML pipelines with Logistic Regression and Random Forest, then tune hyperparameters.

In [ ]:
# Create Logistic Regression pipeline
lr_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000, random_state=42))
])

# Define parameter grid for Logistic Regression
lr_param_grid = {
    'classifier__C': [0.01, 0.1, 1.0, 10.0],
    'classifier__penalty': ['l2'],
    'classifier__solver': ['lbfgs']
}

print("Logistic Regression Pipeline Created:")
print(lr_pipeline)

In [ ]:
# Create Random Forest pipeline
rf_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(random_state=42))
])

# Define parameter grid for Random Forest
rf_param_grid = {
    'classifier__n_estimators': [50, 100, 200],
    'classifier__max_depth': [None, 10, 20],
    'classifier__min_samples_split': [2, 5]
}

print("Random Forest Pipeline Created:")
print(rf_pipeline)

In [ ]:
# Hyperparameter tuning for Logistic Regression
print("Tuning Logistic Regression...")
lr_grid = GridSearchCV(
    lr_pipeline, 
    param_grid=lr_param_grid,
    cv=5,
    scoring='f1',
    n_jobs=-1,
    verbose=1
)

lr_grid.fit(X_train, y_train)

print(f"\nBest Logistic Regression Parameters: {lr_grid.best_params_}")
print(f"Best CV F1-Score: {lr_grid.best_score_:.4f}")

In [ ]:
# Hyperparameter tuning for Random Forest
print("\nTuning Random Forest...")
rf_grid = GridSearchCV(
    rf_pipeline, 
    param_grid=rf_param_grid,
    cv=5,
    scoring='f1',
    n_jobs=-1,
    verbose=1
)

rf_grid.fit(X_train, y_train)

print(f"\nBest Random Forest Parameters: {rf_grid.best_params_}")
print(f"Best CV F1-Score: {rf_grid.best_score_:.4f}")

## 5. Model Evaluation

Compare model performance on the test set.

In [ ]:
# Get best models
lr_best = lr_grid.best_estimator_
rf_best = rf_grid.best_estimator_

# Generate predictions
lr_pred = lr_best.predict(X_test)
rf_pred = rf_best.predict(X_test)

# Calculate metrics
def get_metrics(y_true, y_pred, model_name):
    return {
        'Model': model_name,
        'Accuracy': accuracy_score(y_true, y_pred),
        'Precision': precision_score(y_true, y_pred),
        'Recall': recall_score(y_true, y_pred),
        'F1-Score': f1_score(y_true, y_pred),
        'ROC-AUC': roc_auc_score(y_true, y_pred)
    }

lr_metrics = get_metrics(y_test, lr_pred, 'Logistic Regression')
rf_metrics = get_metrics(y_test, rf_pred, 'Random Forest')

# Display metrics
metrics_df = pd.DataFrame([lr_metrics, rf_metrics])
print("\nModel Comparison (Test Set):")
print(metrics_df.to_string(index=False))

In [ ]:
# Detailed classification reports
print("\n" + "="*60)
print("Logistic Regression - Classification Report")
print("="*60)
print(classification_report(y_test, lr_pred, target_names=['No Churn', 'Churn']))

print("\n" + "="*60)
print("Random Forest - Classification Report")
print("="*60)
print(classification_report(y_test, rf_pred, target_names=['No Churn', 'Churn']))

In [ ]:
# Visualize confusion matrices
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Logistic Regression
cm_lr = confusion_matrix(y_test, lr_pred)
sns.heatmap(cm_lr, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['No Churn', 'Churn'], yticklabels=['No Churn', 'Churn'])
axes[0].set_title('Logistic Regression - Confusion Matrix')
axes[0].set_ylabel('Actual')
axes[0].set_xlabel('Predicted')

# Random Forest
cm_rf = confusion_matrix(y_test, rf_pred)
sns.heatmap(cm_rf, annot=True, fmt='d', cmap='Greens', ax=axes[1],
            xticklabels=['No Churn', 'Churn'], yticklabels=['No Churn', 'Churn'])
axes[1].set_title('Random Forest - Confusion Matrix')
axes[1].set_ylabel('Actual')
axes[1].set_xlabel('Predicted')

plt.tight_layout()
plt.show()

In [ ]:
# ROC Curve comparison
lr_probs = lr_best.predict_proba(X_test)[:, 1]
rf_probs = rf_best.predict_proba(X_test)[:, 1]

fpr_lr, tpr_lr, _ = roc_curve(y_test, lr_probs)
fpr_rf, tpr_rf, _ = roc_curve(y_test, rf_probs)

plt.figure(figsize=(8, 6))
plt.plot(fpr_lr, tpr_lr, label=f'Logistic Regression (AUC = {roc_auc_score(y_test, lr_probs):.4f})', 
         color='blue', linewidth=2)
plt.plot(fpr_rf, tpr_rf, label=f'Random Forest (AUC = {roc_auc_score(y_test, rf_probs):.4f})', 
         color='green', linewidth=2)
plt.plot([0, 1], [0, 1], 'r--', label='Random Classifier', linewidth=1)
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve Comparison')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## 6. Feature Importance (Random Forest)

In [ ]:
# Extract feature importance from Random Forest
rf_classifier = rf_best.named_steps['classifier']
ohe = rf_best.named_steps['preprocessor'].named_transformers_['cat'].named_steps['onehot']

# Get feature names after one-hot encoding
cat_feature_names = ohe.get_feature_names_out(categorical_cols).tolist()
all_feature_names = numerical_cols + cat_feature_names

# Create feature importance dataframe
feature_importance = pd.DataFrame({
    'Feature': all_feature_names,
    'Importance': rf_classifier.feature_importances_
}).sort_values('Importance', ascending=False)

# Plot top 15 features
plt.figure(figsize=(10, 6))
top_features = feature_importance.head(15)
sns.barplot(data=top_features, x='Importance', y='Feature', palette='viridis')
plt.title('Top 15 Important Features (Random Forest)')
plt.tight_layout()
plt.show()

print("Top 10 Most Important Features:")
print(feature_importance.head(10).to_string(index=False))

## 7. Export the Complete Pipeline

Save the best model pipeline using joblib for production use.

In [ ]:
# Determine the best model based on F1-Score
if rf_metrics['F1-Score'] > lr_metrics['F1-Score']:
    best_model = rf_best
    best_model_name = 'Random Forest'
else:
    best_model = lr_best
    best_model_name = 'Logistic Regression'

print(f"Best Model: {best_model_name}")

# Export the complete pipeline
pipeline_filename = f'churn_prediction_pipeline_{best_model_name.replace(" ", "_").lower()}.joblib'
joblib.dump(best_model, pipeline_filename)
print(f"\nComplete pipeline saved to: {pipeline_filename}")

# Save preprocessing pipeline separately for reusability
joblib.dump(preprocessor, 'churn_preprocessor.joblib')
print(f"Preprocessor saved to: churn_preprocessor.joblib")

In [ ]:
# Demonstrate loading and using the saved pipeline
print("\n" + "="*60)
print("Loading and Testing the Saved Pipeline")
print("="*60)

# Load the pipeline
loaded_pipeline = joblib.load(pipeline_filename)
print(f"\nPipeline loaded from: {pipeline_filename}")

# Make prediction on a sample
sample_data = X_test.iloc[:5]
predictions = loaded_pipeline.predict(sample_data)
probabilities = loaded_pipeline.predict_proba(sample_data)[:, 1]

print("\nSample Predictions:")
for idx, (pred, prob) in enumerate(zip(predictions, probabilities)):
    status = "CHURN" if pred == 1 else "NO CHURN"
    print(f"  Sample {idx+1}: {status} (probability: {prob:.4f})")

## Final Summary / Insights

- **Pipeline Architecture**: Complete end-to-end ML pipeline with preprocessing and classification
- **Models Compared**: Logistic Regression vs Random Forest
- **Hyperparameter Tuning**: GridSearchCV with 5-fold cross-validation
- **Best Model**: Exported as a reusable joblib file

### Key Findings
- Random Forest typically outperforms Logistic Regression on this dataset
- Most important features: Contract type, tenure, Monthly charges, Internet service type
- Month-to-month contracts and high monthly charges correlate with higher churn

### Production Readiness
- Pipeline handles missing values automatically
- Handles both numerical and categorical features
- Can be deployed as a service for real-time churn prediction
- Easy to retrain and update with new data